In [ ]:
!pip install transformers datasets scikit-learn pandas tqdm

In [ ]:
from datasets import load_dataset
import pandas as pd

# تحميل IMDb dataset
dataset = load_dataset("imdb", split="test")

# تحويلها إلى DataFrame
df = pd.DataFrame(dataset)

# أخذ 150 positive و 150 negative
positive_df = df[df["label"] == 1].sample(150, random_state=42)
negative_df = df[df["label"] == 0].sample(150, random_state=42)

# دمج البيانات وعمل shuffle
clean_df = pd.concat([positive_df, negative_df])
clean_df = clean_df.sample(frac=1, random_state=42).reset_index(drop=True)

clean_df = clean_df.rename(columns={"label": "true_label"})

clean_df["true_label"] = clean_df["true_label"].map({
    0: "NEGATIVE",
    1: "POSITIVE"
})

clean_df = clean_df[["text", "true_label"]]


clean_df.to_csv("clean_dataset.csv", index=False)

# عرض أول 5 صفوف
clean_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

,text,true_label
0,"The core message is strong, the cast has given...",NEGATIVE
1,This film is about a young man's painful journ...,NEGATIVE
2,"With a title like that, you will be forgiven f...",NEGATIVE
3,I was recently given this film on DVD as a gif...,POSITIVE
4,I was looking forward to this based on the rev...,NEGATIVE


In [ ]:
print(clean_df.shape)
print(clean_df["true_label"].value_counts())

(300, 2)
true_label
NEGATIVE    150
POSITIVE    150
Name: count, dtype: int64


In [ ]:
from transformers import pipeline
from tqdm import tqdm

# تحميل موديل DistilBERT
distilbert = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# قوائم لحفظ النتائج
distilbert_preds = []
distilbert_scores = []

# تشغيل الموديل على كل النصوص
for text in tqdm(clean_df["text"]):

    result = distilbert(text[:512])[0]

    distilbert_preds.append(result["label"])
    distilbert_scores.append(result["score"])

# حفظ النتائج داخل الداتا
clean_df["DistilBERT_pred"] = distilbert_preds
clean_df["DistilBERT_score"] = distilbert_scores

# عرض النتائج
clean_df.head()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

  5%|▌         | 16/300 [00:04<01:09,  4.06it/s]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# القيم الحقيقية
y_true = clean_df["true_label"]

# توقعات الموديل
y_pred = clean_df["DistilBERT_pred"]

# حساب المقاييس
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label="POSITIVE")
recall = recall_score(y_true, y_pred, pos_label="POSITIVE")
f1 = f1_score(y_true, y_pred, pos_label="POSITIVE")

# عرض النتائج
print("DistilBERT Results")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

DistilBERT Results
Accuracy: 0.81
Precision: 0.8252
Recall: 0.7867
F1-score: 0.8055


In [ ]:
from transformers import pipeline
from tqdm import tqdm

# تحميل موديل RoBERTa
roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)

# دالة توحيد الليبلات
def unify_roberta_label(label):

    if label == "LABEL_2":
        return "POSITIVE"

    else:
        return "NEGATIVE"

# قوائم حفظ النتائج
roberta_preds = []
roberta_scores = []

# تشغيل الموديل
for text in tqdm(clean_df["text"]):

    result = roberta(text[:512])[0]

    unified_label = unify_roberta_label(result["label"])

    roberta_preds.append(unified_label)
    roberta_scores.append(result["score"])

# حفظ النتائج
clean_df["RoBERTa_pred"] = roberta_preds
clean_df["RoBERTa_score"] = roberta_scores

# عرض البيانات
clean_df.head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 300/300 [02:51<00:00,  1.75it/s]


,text,true_label,DistilBERT_pred,DistilBERT_score,RoBERTa_pred,RoBERTa_score
0,"The core message is strong, the cast has given...",NEGATIVE,NEGATIVE,0.997454,NEGATIVE,0.386150
1,This film is about a young man's painful journ...,NEGATIVE,POSITIVE,0.975371,NEGATIVE,0.501295
2,"With a title like that, you will be forgiven f...",NEGATIVE,NEGATIVE,0.994927,NEGATIVE,0.636101
3,I was recently given this film on DVD as a gif...,POSITIVE,POSITIVE,0.998675,POSITIVE,0.855483
4,I was looking forward to this based on the rev...,NEGATIVE,NEGATIVE,0.999450,NEGATIVE,0.812834


In [ ]:
# توقعات RoBERTa
y_pred = clean_df["RoBERTa_pred"]

# حساب المقاييس
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label="POSITIVE")
recall = recall_score(y_true, y_pred, pos_label="POSITIVE")
f1 = f1_score(y_true, y_pred, pos_label="POSITIVE")

# عرض النتائج
print("RoBERTa Results")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

RoBERTa Results
Accuracy: 0.7667
Precision: 0.8922
Recall: 0.6067
F1-score: 0.7222


In [ ]:
from transformers import pipeline
from tqdm import tqdm

# تحميل موديل BERT
bert = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# دالة توحيد التصنيفات
def unify_bert_label(label):

    if label in ["4 stars", "5 stars"]:
        return "POSITIVE"

    else:
        return "NEGATIVE"

# قوائم حفظ النتائج
bert_preds = []
bert_scores = []

# تشغيل الموديل
for text in tqdm(clean_df["text"]):

    result = bert(text[:512])[0]

    unified_label = unify_bert_label(result["label"])

    bert_preds.append(unified_label)
    bert_scores.append(result["score"])

# حفظ النتائج
clean_df["BERT_pred"] = bert_preds
clean_df["BERT_score"] = bert_scores

# عرض البيانات
clean_df.head()

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

100%|██████████| 300/300 [02:39<00:00,  1.89it/s]


,text,true_label,DistilBERT_pred,DistilBERT_score,RoBERTa_pred,RoBERTa_score,BERT_pred,BERT_score
0,"The core message is strong, the cast has given...",NEGATIVE,NEGATIVE,0.997454,NEGATIVE,0.386150,NEGATIVE,0.453077
1,This film is about a young man's painful journ...,NEGATIVE,POSITIVE,0.975371,NEGATIVE,0.501295,POSITIVE,0.405105
2,"With a title like that, you will be forgiven f...",NEGATIVE,NEGATIVE,0.994927,NEGATIVE,0.636101,NEGATIVE,0.479811
3,I was recently given this film on DVD as a gif...,POSITIVE,POSITIVE,0.998675,POSITIVE,0.855483,NEGATIVE,0.340471
4,I was looking forward to this based on the rev...,NEGATIVE,NEGATIVE,0.999450,NEGATIVE,0.812834,NEGATIVE,0.613582


In [ ]:
# توقعات BERT
y_pred = clean_df["BERT_pred"]

# حساب المقاييس
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label="POSITIVE")
recall = recall_score(y_true, y_pred, pos_label="POSITIVE")
f1 = f1_score(y_true, y_pred, pos_label="POSITIVE")

# عرض النتائج
print("BERT Results")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

BERT Results
Accuracy: 0.7633
Precision: 0.8624
Recall: 0.6267
F1-score: 0.7259


In [ ]:
results_df = pd.DataFrame({
    "Model": ["DistilBERT", "RoBERTa", "BERT"],

    "Accuracy": [0.81, 0.7667, accuracy],

    "Precision": [0.8252, 0.8922, precision],

    "Recall": [0.7867, 0.6067, recall],

    "F1-score": [0.8055, 0.7222, f1]
})

results_df

,Model,Accuracy,Precision,Recall,F1-score
0,DistilBERT,0.810000,0.825200,0.786700,0.805500
1,RoBERTa,0.766700,0.892200,0.606700,0.722200
2,BERT,0.763333,0.862385,0.626667,0.725869


In [ ]:
# حفظ كل التوقعات
clean_df.to_csv("predictions.csv", index=False)

# حفظ جدول المقارنة
results_df.to_csv("model_results.csv", index=False)

print("Files saved successfully!")

Files saved successfully!


In [ ]:
from google.colab import files

files.download("clean_dataset.csv")
files.download("predictions.csv")
files.download("model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
with pd.ExcelWriter("NLP_Project_Files.xlsx") as writer:
    clean_df.to_excel(writer, sheet_name="Predictions", index=False)
    results_df.to_excel(writer, sheet_name="Results", index=False)

print("Excel file created!")

Excel file created!


In [ ]:
from google.colab import files

files.download("NLP_Project_Files.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile app.py

import streamlit as st
from transformers import pipeline

st.set_page_config(
    page_title="Sentiment Classification App",
    page_icon="💬",
    layout="centered"
)

st.title("Sentiment Classification System")
st.write("Classify text sentiment using three pretrained Hugging Face models.")

MODELS = {
    "DistilBERT": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa": "cardiffnlp/twitter-roberta-base-sentiment",
    "BERT": "nlptown/bert-base-multilingual-uncased-sentiment"
}

@st.cache_resource
def load_model(model_name):
    return pipeline("sentiment-analysis", model=MODELS[model_name])

def unify_label(model_name, label):
    if model_name == "DistilBERT":
        return label

    if model_name == "RoBERTa":
        return "POSITIVE" if label == "LABEL_2" else "NEGATIVE"

    if model_name == "BERT":
        return "POSITIVE" if label in ["4 stars", "5 stars"] else "NEGATIVE"

sentence = st.text_area("Enter a sentence:")

model_choice = st.selectbox(
    "Choose a model:",
    ["DistilBERT", "RoBERTa", "BERT"]
)

if st.button("Predict"):
    if sentence.strip() == "":
        st.warning("Please enter a sentence first.")
    else:
        classifier = load_model(model_choice)
        result = classifier(sentence[:512])[0]

        final_label = unify_label(model_choice, result["label"])
        confidence = result["score"] * 100

        st.subheader("Prediction Result")
        st.success(f"Predicted Label: {final_label}")
        st.write(f"Confidence Score: {confidence:.2f}%")

Overwriting app.py


In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 54.3 MB/s eta 0:00:00


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://eighty-states-raise.loca.lt
2026-05-13 13:46:39.545 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://162.222.177.131:8501

y
Loading weights: 100% 104/104 [00:00<00:00, 859.48it/s, Materializing param=pre_classifier.weight]
Loading weights: 100% 201/201 [00:00<00:00, 884.55it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100% 201/201 [00:00<00:00, 1257.58it/s, Materializing param=classifier.weight]
